# Introduction

In these exercises we'll look at how pandas represents data types and handles missing values.

Run the code cell below to load the data before running the exercises.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 150

country_pool = ['Italy', 'Portugal', 'US', 'Spain', 'France', 'Australia']
variety_pool = ['Pinot Noir', 'Chardonnay', 'Cabernet Sauvignon', 'Riesling',
                'Sangiovese', 'Shiraz']
region_pool = ['Napa Valley', 'Willamette Valley', 'Tuscany', 'Douro',
               'Bordeaux', 'Barossa Valley', 'Marlborough', 'Rioja']

countries = list(np.random.choice(country_pool, n))
varieties = list(np.random.choice(variety_pool, n))
points = list(np.random.randint(82, 98, n))

raw_prices = np.random.uniform(10, 200, n)
raw_regions = list(np.random.choice(region_pool, n))

# Introduce missing values: ~20% of prices, ~30% of region_1
prices = [None if i % 5 == 0 else round(float(raw_prices[i]), 1) for i in range(n)]
regions = [None if i % 3 == 0 else raw_regions[i] for i in range(n)]

reviews = pd.DataFrame({
    'country': countries,
    'description': [f'Wine description for review {i}.' for i in range(n)],
    'designation': [None if i % 4 == 0 else f'Label {i % 15}' for i in range(n)],
    'points': points,
    'price': prices,
    'province': [f'Province {i % 8}' for i in range(n)],
    'region_1': regions,
    'region_2': [None if i % 4 == 0 else f'Sub-region {i % 5}' for i in range(n)],
    'taster_name': [None if i % 5 == 0 else f'Taster {i % 6}' for i in range(n)],
    'title': [f'Winery {i % 20} {2010 + i % 10} Vintage Wine' for i in range(n)],
    'variety': varieties,
    'winery': [f'Winery {i % 20}' for i in range(n)],
})

pd.set_option('display.max_rows', 5)
print(f'Loaded {len(reviews)} wine reviews with {reviews.shape[1]} columns.')
print(f'Missing prices: {sum(1 for p in prices if p is None)}')
print(f'Missing region_1: {sum(1 for r in regions if r is None)}')
print('Setup complete.')
reviews.head()

Loaded 150 wine reviews with 12 columns.
Missing prices: 30
Missing region_1: 50
Setup complete.


,country,description,designation,points,price,province,region_1,region_2,taster_name,title,variety,winery
0,Spain,Wine description for review 0.,NaN,97,NaN,Province 0,NaN,NaN,NaN,Winery 0 2010 Vintage Wine,Sangiovese,Winery 0
1,France,Wine description for review 1.,Label 1,95,15.2,Province 1,Willamette Valley,Sub-region 1,Taster 1,Winery 1 2011 Vintage Wine,Sangiovese,Winery 1
2,US,Wine description for review 2.,Label 2,90,120.0,Province 2,Willamette Valley,Sub-region 2,Taster 2,Winery 2 2012 Vintage Wine,Cabernet Sauvignon,Winery 2
3,France,Wine description for review 3.,Label 3,84,93.3,Province 3,NaN,Sub-region 3,Taster 3,Winery 3 2013 Vintage Wine,Sangiovese,Winery 3
4,France,Wine description for review 4.,NaN,90,137.7,Province 4,Napa Valley,NaN,Taster 4,Winery 4 2014 Vintage Wine,Riesling,Winery 4


# Exercises

## 1.

What is the data type of the `points` column in the dataset?

In [2]:
dtype = reviews.points.dtype
dtype

dtype('int32')

**Key insight:** Every pandas Series has a single `.dtype`. Common types:

| dtype | Pandas type | Typical use |
|---|---|---|
| `int64` | integer | counts, points, IDs |
| `float64` | float | prices, ratios (also used when column has NaN) |
| `object` | string / mixed | text, categories |
| `bool` | boolean | True/False flags |

A column of integers with even one NaN becomes `float64` — because Python's `int` can't represent NaN but `float` can (as `nan`).

## 2.

Create a Series from entries in the `points` column, but convert the entries to strings. Assign the result to the variable `point_strings`.

In [3]:
point_strings = reviews.points.astype('str')
print(point_strings.dtype)
point_strings.head()

str


0    97
1    95
2    90
3    84
4    90
Name: points, dtype: str

**Key insight:** `.astype(dtype)` converts a whole Series to a new type in one vectorised call. Common conversions:

```python
series.astype('str')      # → object (string)
series.astype('float64')  # → float
series.astype('int64')    # → int (fails if NaN present — use pd.Int64Dtype() for nullable int)
series.astype('category') # → memory-efficient for low-cardinality columns
```

Note: `astype` does **not** modify the original Series — it returns a new one.

## 3.

How many reviews in the dataset are missing a `price`? Create a variable `n_missing_prices` with the count.

In [4]:
n_missing_prices = reviews.price.isnull().sum()
print(f'{n_missing_prices} reviews are missing a price.')

30 reviews are missing a price.


**Key insight:** The missing-value toolkit:

| Method | Returns | Use when |
|---|---|---|
| `.isnull()` | boolean Series (True = missing) | detecting NaN |
| `.notnull()` | boolean Series (True = present) | filtering to valid rows |
| `.isnull().sum()` | count of NaN | auditing data quality |
| `.isnull().mean()` | fraction of NaN | comparing missingness across columns |

`pd.isna()` is an alias for `.isnull()` — both are identical.

## 4.

What are the most common wine-producing regions? Create a Series `reviews_per_region` mapping each region to the number of reviews from that region. Replace missing values in the `region_1` column with `'Unknown'`.

In [5]:
reviews_per_region = reviews.region_1.fillna('Unknown').value_counts()
reviews_per_region

region_1
Unknown        50
Bordeaux       18
               ..
Marlborough     9
Tuscany         9
Name: count, Length: 9, dtype: int64

**Key insight:** Missing value strategies:

| Method | Use case |
|---|---|
| `.fillna(value)` | replace NaN with a constant (e.g., `'Unknown'`, `0`) |
| `.fillna(series.mean())` | replace NaN with column mean (numeric imputation) |
| `.fillna(method='ffill')` | forward-fill — carry last valid value forward (time series) |
| `.dropna()` | remove rows that contain NaN |

Chaining: `.fillna('Unknown').value_counts()` works because `.fillna()` returns a **new** Series — the original DataFrame is unchanged.

---
## Session Summary — Day 5 Pandas

| Exercise | Topic | Key Concept |
|----------|-------|-------------|
| 1 | Check column dtype | `.dtype` — int/float/object/bool; int column with NaN becomes float64 |
| 2 | Type conversion | `.astype('str')` — vectorised, returns new Series, doesn't mutate original |
| 3 | Count missing values | `.isnull().sum()` — True counts as 1; `.isnull().mean()` gives fraction |
| 4 | Fill and aggregate | `.fillna('Unknown').value_counts()` — replace NaN before aggregating |